# China — Monthly Notifiable Disease Overviews, CNIC Influenza & COVID Reports

The `china_cdc` accessor covers **four complementary mainland-China sources**. The
journal side (China CDC Weekly, discovered via CrossRef) is demonstrated in
*15_China_CDC_Weekly_Surveillance.ipynb*; this notebook focuses on the
direct-report pipelines:

- **NDCPA monthly overviews** (国家疾病预防控制局, mirrored on chinacdc.cn): official
  per-disease case/death tables for all statutory Class A/B/C diseases **plus** the
  priority-monitored non-notifiable diseases (e.g. chickenpox, liver fluke disease)
- **CNIC weekly influenza reports** (Chinese National Influenza Center): sentinel ILI%
  for southern/northern provinces and laboratory positivity by type/subtype
- **Monthly national COVID-19 situation reports** (chinacdc.cn)

All listings are server-rendered HTML, so no CrossRef or JavaScript rendering is needed.

## 1. Setup

In [ ]:
from epidatasets.sources.china_cdc import ChinaCDCAccessor

ccdc = ChinaCDCAccessor()  # listings fetched live; PDFs cached with 365-day TTL

ccdc.list_notifiable_diseases().head(8)

## 2. Monthly notifiable disease overviews

Each month the NDCPA publishes a per-disease table (法定传染病疫情概况). The accessor
scrapes the chinacdc.cn mirror listing and parses the Chinese table into the same
canonical schema used for the China CDC Weekly reports, mapping Chinese disease names
to codes via the accessor's built-in dictionary.

In [ ]:
overview = ccdc.list_monthly_overviews(year=2026)
overview

In [ ]:
jul = ccdc.get_monthly_overview(2026, 7)
print(f"{len(jul)} disease rows for 2026-07")
jul[jul["category"] == "Class A"]

In [ ]:
# Class B totals and the top-level diseases (hepatitis subtypes are flagged as sub-items)
class_b = jul[jul["category"] == "Class B"]
class_b[~class_b["is_subitem"]].nlargest(5, "cases")[
    ["disease_code", "disease_cn", "cases", "deaths"]
]

### The unique extra: priority-monitored non-notifiable diseases

Since January 2026 the monthly overviews also include a "key monitored" section with
diseases that are **not** statutorily notifiable — data unavailable anywhere else
in this package.

In [ ]:
monitored = jul[jul["category"] == "Monitored (non-notifiable)"]
monitored[["disease_code", "disease_cn", "cases", "deaths"]]

### Monthly and annual summaries

In [ ]:
ccdc.get_monthly_summary(2026, 7)

In [ ]:
# aggregates every available month of 2026 (fetches each monthly page)
annual = ccdc.get_annual_summary(2026)
annual.nlargest(8, "total_cases")[["disease_cn", "category", "total_cases", "total_deaths", "months_reported"]]

## 3. CNIC weekly influenza surveillance

The Chinese National Influenza Center publishes an English weekly report per week.
The accessor lists the reports, downloads the PDFs (cached) and parses the ILI%
sentinel rates and the laboratory Table 1.

In [ ]:
reports = ccdc.list_cnic_weekly_reports(year=2025, max_pages=3)
reports.tail()

In [ ]:
flu = ccdc.get_influenza_surveillance(year=2025, weeks=[35, 36, 37])
flu[["year", "week", "period", "ili_percent_south", "ili_percent_north",
     "specimens_tested", "positivity_rate", "ili_outbreaks"]]

In [ ]:
week37 = flu.iloc[-1]
import pandas as pd

pd.DataFrame(week37["by_type"]).T  # Table 1: laboratory detections by type/subtype

## 4. Monthly COVID-19 situation reports

chinacdc.cn publishes a monthly national COVID-19 report; the accessor extracts the
headline counts (handling the 万 = 10k unit used in the prose).

In [ ]:
covid = ccdc.get_covid_updates()
covid[["year", "month", "new_cases", "severe_cases", "new_deaths"]].tail(6)

## 5. Putting it together — influenza on both sides of the border

The CNIC series (mainland) and the Hong Kong Flu Express (`hk_chp` accessor) can be
compared directly.

In [ ]:
# requires network; both series are cached after the first fetch
from epidatasets.sources.hk_chp import HongKongCHPAccessor

hk = HongKongCHPAccessor()
hk_flu = hk.get_influenza_positivity(year=2025).set_index("week_start")
cn_ili = flu.dropna(subset=["ili_percent_south"]).assign(
    week_start=pd.to_datetime(flu["period"].str.split(" to ").str[0] + ", 2025",
                              format="%B %d, %Y")
).set_index("week_start")

ax = hk_flu["influenza_positivity"].plot(figsize=(11, 4), label="HK positivity", legend=True)
cn_ili.plot(x="week_start", y="ili_percent_south", secondary_y=True, ax=ax,
            color="firebrick", legend=True, label="Mainland ILI% (south)");